In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.4 GQA / MQA

At inference the KV cache dominates memory. **Grouped-query attention** shares one
set of key/value heads across several query heads; **multi-query** is the extreme
of a single KV head. Same query expressivity, a fraction of the KV memory.

In [ ]:
from pocketlm import PocketLM, PocketLMConfig

def kv_params(n_kv):
    cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=1, n_head=4,
                         n_embd=32, n_kv_head=n_kv)
    attn = PocketLM(cfg).blocks[0].attn
    return sum(p.numel() for p in attn.c_kv.parameters())

print("full (4 KV heads):", kv_params(4))
print("GQA  (2 KV heads):", kv_params(2))
print("MQA  (1 KV head) :", kv_params(1))

Halving the KV heads halves the KV projection (and the cache it feeds). The model
still works, the no-future-leak invariant still holds, the memory bill drops.

In [ ]:
assert kv_params(1) < kv_params(2) < kv_params(4)
cfg = PocketLMConfig(vocab_size=20, block_size=16, n_layer=1, n_head=4, n_embd=32,
                     n_kv_head=1)
logits, _ = PocketLM(cfg)(torch.randint(0, 20, (1, 8)))
assert logits.shape == (1, 8, 20)